# End-to-End Medallion Reconciliation — Source → Bronze → Silver → Gold

**Purpose:** comprehensive data-quality and metric audit proving every record and
dollar amount is accounted for across the entire pipeline.

## Three audit phases

### Phase 1 — Source → Bronze parity
Confirms `bronze_orders` and `bronze_transactions` each contain exactly **1,000,000**
rows matching the raw CSV extracts in the UC volume, with all audit columns
(`_ingested_at`, `_source_file`, `_batch_id`, `_row_hash`) present and non-null.

### Phase 2 — Silver accounting invariant
Proves the strict accounting identity for both datasets:

$$\text{Bronze rows} = \text{accepted Silver rows} + \text{quarantined rows} + \text{duplicate losers}$$

Row-level verification uses the `_row_hash` audit identifier to guarantee **zero
overlap** between accepted Silver, quarantined, and duplicate-loser buckets, and
that their union covers every Bronze row exactly once.

### Phase 3 — Gold → Silver financial reconciliation
Rebuilds every Gold mart's revenue and order-count logic independently from the
trusted Silver tables and asserts exact agreement with the materialised Gold
tables — producing the verified Databricks SQL baseline figures needed to confirm
the AI/BI dashboard and Power BI report under identical filters.

**Revenue rules (applied consistently across all marts):**
- Eligible: `order_status = 'shipped'` AND `txn_status = 'success'`
- Gross eligible revenue = SUM(success txn amounts on shipped orders, in USD via FX)
- Net revenue = gross eligible − refunds on shipped orders
- FX rates to USD: USD=1, EUR=1.08, GBP=1.27, INR=0.012, JPY=0.0067, CAD=0.73, AUD=0.66, SGD=0.74, BDT=0.0091; ELSE NULL (unmapped excluded)

This notebook writes nothing to the data tables and raises an error if any
assertion fails.

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("schema", "ecom_poc", "Schema")
dbutils.widgets.text("source_catalog", "workspace", "Source catalog (volume)")
dbutils.widgets.text("source_schema", "ecom_poc", "Source schema (volume)")
dbutils.widgets.text("source_volume", "raw_files", "Source volume")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
SOURCE_CATALOG = dbutils.widgets.get("source_catalog").strip()
SOURCE_SCHEMA = dbutils.widgets.get("source_schema").strip()
SOURCE_VOLUME = dbutils.widgets.get("source_volume").strip()
VOLUME_PATH = f"/Volumes/{SOURCE_CATALOG}/{SOURCE_SCHEMA}/{SOURCE_VOLUME}"

def fq(name):
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"

EXPECTED_ROWS = 1_000_000

# Static FX rates to USD — must match the Gold notebook exactly
FX = ("CASE currency "
      "WHEN 'USD' THEN 1.0 "
      "WHEN 'EUR' THEN 1.08 "
      "WHEN 'GBP' THEN 1.27 "
      "WHEN 'INR' THEN 0.012 "
      "WHEN 'JPY' THEN 0.0067 "
      "WHEN 'CAD' THEN 0.73 "
      "WHEN 'AUD' THEN 0.66 "
      "WHEN 'SGD' THEN 0.74 "
      "WHEN 'BDT' THEN 0.0091 "
      "ELSE NULL END")

DATASETS = {
    "orders": {
        "bronze": fq("bronze_orders"),
        "silver": fq("silver_orders"),
        "quarantine": fq("quarantine_orders"),
        "source_file": "orders.csv",
        "pk": "order_id",
        "dup_label": "DUPLICATE_ORDER_ID",
    },
    "transactions": {
        "bronze": fq("bronze_transactions"),
        "silver": fq("silver_transactions"),
        "quarantine": fq("quarantine_transactions"),
        "source_file": "transactions.csv",
        "pk": "txn_id",
        "dup_label": "DUPLICATE_TXN_ID",
    },
}

all_passed = True
print(f"Catalog: {CATALOG}.{SCHEMA}")
print(f"Volume:  {VOLUME_PATH}")

In [0]:
required_tables = [
    "bronze_orders", "bronze_transactions",
    "silver_orders", "silver_transactions",
    "quarantine_orders", "quarantine_transactions",
    "gold_daily_sales", "gold_revenue_breakdown",
    "gold_payment_success", "gold_top_customers",
]

missing = []
for t in required_tables:
    if not spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{t}"):
        missing.append(t)

assert not missing, f"Missing tables — run Bronze, Silver, and Gold first: {missing}"
print(f"\u2713 All {len(required_tables)} tables exist in {CATALOG}.{SCHEMA}.")

In [0]:
from pyspark.sql import functions as F

AUDIT_COLS = ["_ingested_at", "_source_file", "_batch_id", "_row_hash"]

print("=" * 70)
print("PHASE 1 — SOURCE \u2192 BRONZE PARITY")
print("=" * 70)

for name, d in DATASETS.items():
    print(f"\n--- {name} ---")

    csv_path = f"{VOLUME_PATH}/{d['source_file']}"
    csv_count = spark.read.option("header", "true").option("inferSchema", "false").csv(csv_path).count()
    bronze_count = spark.table(d["bronze"]).count()

    bronze_cols = spark.table(d["bronze"]).columns
    missing_audit = [c for c in AUDIT_COLS if c not in bronze_cols]
    cols_ok = len(missing_audit) == 0

    if cols_ok:
        null_counts = (spark.table(d["bronze"])
            .agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in AUDIT_COLS])
            .collect()[0])
        nulls_ok = all(null_counts[c] == 0 for c in AUDIT_COLS)
    else:
        nulls_ok = False

    count_ok = (csv_count == bronze_count == EXPECTED_ROWS)
    table_pass = count_ok and cols_ok and nulls_ok
    all_passed = all_passed and table_pass

    print(f"  Source CSV rows:     {csv_count:,}")
    print(f"  Bronze table rows:   {bronze_count:,}")
    print(f"  Expected:            {EXPECTED_ROWS:,}")
    print(f"  CSV == Bronze == 1M: {'\u2713 PASS' if count_ok else '\u2717 FAIL'}")
    print(f"  Audit cols present:  {'\u2713 PASS' if cols_ok else '\u2717 FAIL — missing: ' + str(missing_audit)}")
    print(f"  Audit cols non-null: {'\u2713 PASS' if nulls_ok else '\u2717 FAIL'}")
    print(f"  OVERALL:             {'\u2713 PASS' if table_pass else '\u2717 FAIL'}")

    if not table_pass:
        raise AssertionError(f"{name}: Source\u2192Bronze parity check FAILED")

print(f"\n{'=' * 70}")
print(f"PHASE 1 RESULT: {'\u2713 ALL CHECKS PASSED' if all_passed else '\u2717 FAILURES'}")
print("=" * 70)

In [0]:
print("=" * 70)
print("PHASE 2 — SILVER ACCOUNTING INVARIANT")
print("  Bronze = Accepted Silver + Quarantined + Duplicate Losers")
print("=" * 70)

for name, d in DATASETS.items():
    print(f"\n--- {name} ---")

    bronze_cnt = spark.table(d["bronze"]).count()
    silver_cnt = spark.table(d["silver"]).count()

    quar_df = spark.table(d["quarantine"])
    quar_cnt = quar_df.count()
    dup_cnt = quar_df.filter(F.col("rejection_reason") == d["dup_label"]).count()
    valfail_cnt = quar_cnt - dup_cnt

    inv_ok = (bronze_cnt == silver_cnt + quar_cnt)
    three_ok = (bronze_cnt == silver_cnt + valfail_cnt + dup_cnt)

    overlap = (spark.table(d["silver"]).select("_row_hash")
               .intersect(quar_df.select("_row_hash"))
               .count())

    coverage = (spark.table(d["silver"]).select("_row_hash")
               .union(quar_df.select("_row_hash"))
               .distinct().count())
    coverage_ok = (coverage == bronze_cnt)

    dup_silver_hash = spark.sql(
        f"SELECT _row_hash FROM {d['silver']} GROUP BY _row_hash HAVING COUNT(*) > 1"
    ).count()

    dup_quar_hash = spark.sql(
        f"SELECT _row_hash FROM {d['quarantine']} GROUP BY _row_hash HAVING COUNT(*) > 1"
    ).count()

    reasons = (quar_df.groupBy("rejection_reason")
               .count().orderBy(F.desc("count")).limit(5).collect())

    table_pass = all([inv_ok, three_ok, overlap == 0, coverage_ok,
                      dup_silver_hash == 0, dup_quar_hash == 0])
    all_passed = all_passed and table_pass

    print(f"  Bronze rows:                  {bronze_cnt:,}")
    print(f"  Accepted Silver:              {silver_cnt:,}")
    print(f"  Quarantine (total):           {quar_cnt:,}")
    print(f"    Validation failures:        {valfail_cnt:,}")
    print(f"    Duplicate losers:            {dup_cnt:,}")
    print(f"  Bronze = Silver + Quarantine:         {'\u2713 PASS' if inv_ok else '\u2717 FAIL'}")
    print(f"  Bronze = Silver + ValFail + Dup:       {'\u2713 PASS' if three_ok else '\u2717 FAIL'}")
    print(f"  Zero _row_hash overlap (Sil/Quar):     {'\u2713 PASS (0 shared)' if overlap == 0 else '\u2717 FAIL (' + str(overlap) + ')'}")
    if coverage_ok:
        print(f"  Full coverage (distinct _row_hash):    \u2713 PASS ({coverage:,} = {bronze_cnt:,})")
    else:
        print(f"  Full coverage (distinct _row_hash):    \u2717 FAIL ({coverage:,} != {bronze_cnt:,})")
    print(f"  No dup _row_hash in Silver:            {'\u2713 PASS' if dup_silver_hash == 0 else '\u2717 FAIL (' + str(dup_silver_hash) + ')'}")
    print(f"  No dup _row_hash in Quarantine:        {'\u2713 PASS' if dup_quar_hash == 0 else '\u2717 FAIL (' + str(dup_quar_hash) + ')'}")
    print(f"  OVERALL:                               {'\u2713 PASS' if table_pass else '\u2717 FAIL'}")
    print(f"  Top rejection reasons:")
    for r in reasons:
        print(f"    {r['rejection_reason']:<45s} {r['count']:>8,}")

    if not table_pass:
        raise AssertionError(f"{name}: Silver accounting invariant FAILED")

print(f"\n{'=' * 70}")
print(f"PHASE 2 RESULT: {'\u2713 ALL CHECKS PASSED' if all_passed else '\u2717 FAILURES'}")
print("=" * 70)

In [0]:
print("=" * 70)
print("PHASE 3 — GOLD \u2192 SILVER FINANCIAL RECONCILIATION")
print("=" * 70)

# --- Rebuild gold logic independently from silver tables ---

# Step 1: Convert transaction amounts to USD (same FX as Gold notebook)
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW recon_txn_converted AS
SELECT order_id, payment_method, txn_status,
       ROUND(amount * {FX}, 2) AS amount_usd
FROM {fq('silver_transactions')}
""")

# Step 2: Aggregate transactions per order_id BEFORE join (prevents inflation)
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW recon_order_revenue AS
SELECT
    o.order_id, o.customer_id, o.order_date, o.category, o.country, o.order_status,
    (o.order_status = 'shipped')    AS is_eligible,
    (o.order_status = 'cancelled')  AS is_cancelled,
    CASE WHEN o.order_status = 'shipped'
         THEN COALESCE(p.paid_amount, 0) ELSE 0 END AS eligible_revenue,
    CASE WHEN o.order_status = 'shipped'
         THEN COALESCE(p.paid_amount, 0) - COALESCE(p.refunded_amount, 0)
         ELSE 0 END AS net_revenue
FROM {fq('silver_orders')} o
LEFT JOIN (
    SELECT order_id,
        SUM(CASE WHEN txn_status = 'success'  THEN amount_usd ELSE 0 END) AS paid_amount,
        SUM(CASE WHEN txn_status = 'refunded' THEN amount_usd ELSE 0 END) AS refunded_amount
    FROM recon_txn_converted GROUP BY order_id
) p ON o.order_id = p.order_id
""")

recon_rows = spark.sql("SELECT COUNT(*) FROM recon_order_revenue").collect()[0][0]
silver_orders_cnt = spark.table(fq("silver_orders")).count()
print(f"\nRebuilt order_revenue from silver: {recon_rows:,} rows (silver_orders: {silver_orders_cnt:,})")
assert recon_rows == silver_orders_cnt, "order_revenue row count != silver_orders"

# --- 3a. gold_daily_sales reconciliation ---
spark.sql("""
CREATE OR REPLACE TEMP VIEW recon_daily AS
SELECT
    order_date,
    CAST(COUNT(*) AS BIGINT) AS order_count,
    CAST(SUM(CASE WHEN is_eligible THEN 1 ELSE 0 END) AS BIGINT) AS eligible_order_count,
    CAST(SUM(CASE WHEN is_cancelled THEN 1 ELSE 0 END) AS BIGINT) AS cancelled_order_count,
    CAST(SUM(CASE WHEN is_cancelled THEN 1 ELSE 0 END) / COUNT(*) AS DECIMAL(9,6)) AS cancellation_rate,
    CAST(SUM(eligible_revenue) AS DECIMAL(28,2)) AS eligible_revenue,
    CAST(SUM(net_revenue) AS DECIMAL(28,2)) AS net_revenue
FROM recon_order_revenue GROUP BY order_date
""")

daily_diff = spark.sql(f"""
    (SELECT order_date, order_count, eligible_order_count, cancelled_order_count,
            cancellation_rate, eligible_revenue, net_revenue FROM recon_daily
     EXCEPT
     SELECT order_date, order_count, eligible_order_count, cancelled_order_count,
            cancellation_rate, eligible_revenue, net_revenue FROM {fq('gold_daily_sales')})
    UNION ALL
    (SELECT order_date, order_count, eligible_order_count, cancelled_order_count,
            cancellation_rate, eligible_revenue, net_revenue FROM {fq('gold_daily_sales')}
     EXCEPT
     SELECT order_date, order_count, eligible_order_count, cancelled_order_count,
            cancellation_rate, eligible_revenue, net_revenue FROM recon_daily)
""").count()

daily_ok = (daily_diff == 0)
all_passed = all_passed and daily_ok
print(f"\n--- gold_daily_sales ---")
print(f"  Rebuilt rows:  {spark.sql('SELECT COUNT(*) FROM recon_daily').collect()[0][0]:,}")
print(f"  Gold rows:     {spark.table(fq('gold_daily_sales')).count():,}")
print(f"  Row-by-row match: {'\u2713 PASS' if daily_ok else '\u2717 FAIL (' + str(daily_diff) + ')'}")
if not daily_ok:
    raise AssertionError("gold_daily_sales reconciliation FAILED")

# --- 3b. gold_revenue_breakdown reconciliation ---
spark.sql("""
CREATE OR REPLACE TEMP VIEW recon_breakdown AS
WITH grp AS (
    SELECT category, country,
        CAST(COUNT(*) AS BIGINT) AS order_count,
        CAST(COUNT(DISTINCT customer_id) AS BIGINT) AS customer_count,
        CAST(SUM(eligible_revenue) AS DECIMAL(28,2)) AS eligible_revenue,
        CAST(SUM(net_revenue) AS DECIMAL(28,2)) AS net_revenue
    FROM recon_order_revenue GROUP BY category, country
)
SELECT category, country, order_count, customer_count, eligible_revenue, net_revenue,
    CAST(eligible_revenue / NULLIF(SUM(eligible_revenue) OVER (), 0) AS DECIMAL(9,6)) AS revenue_share
FROM grp
""")

breakdown_diff = spark.sql(f"""
    (SELECT category, country, order_count, customer_count, eligible_revenue, net_revenue, revenue_share FROM recon_breakdown
     EXCEPT
     SELECT category, country, order_count, customer_count, eligible_revenue, net_revenue, revenue_share FROM {fq('gold_revenue_breakdown')})
    UNION ALL
    (SELECT category, country, order_count, customer_count, eligible_revenue, net_revenue, revenue_share FROM {fq('gold_revenue_breakdown')}
     EXCEPT
     SELECT category, country, order_count, customer_count, eligible_revenue, net_revenue, revenue_share FROM recon_breakdown)
""").count()

breakdown_ok = (breakdown_diff == 0)
all_passed = all_passed and breakdown_ok
print(f"\n--- gold_revenue_breakdown ---")
print(f"  Rebuilt rows:  {spark.sql('SELECT COUNT(*) FROM recon_breakdown').collect()[0][0]:,}")
print(f"  Gold rows:     {spark.table(fq('gold_revenue_breakdown')).count():,}")
print(f"  Row-by-row match: {'\u2713 PASS' if breakdown_ok else '\u2717 FAIL (' + str(breakdown_diff) + ')'}")
if not breakdown_ok:
    raise AssertionError("gold_revenue_breakdown reconciliation FAILED")

In [0]:
# --- 3c. gold_payment_success reconciliation ---
spark.sql("""
CREATE OR REPLACE TEMP VIEW recon_payment AS
SELECT payment_method,
    CAST(COUNT(*) AS BIGINT) AS transaction_count,
    CAST(SUM(CASE WHEN txn_status = 'success' THEN 1 ELSE 0 END) AS BIGINT) AS successful_count,
    CAST(SUM(CASE WHEN txn_status = 'success' THEN 1 ELSE 0 END) / COUNT(*) AS DECIMAL(9,6)) AS success_rate,
    CAST(SUM(amount_usd) AS DECIMAL(28,2)) AS total_amount_usd
FROM recon_txn_converted GROUP BY payment_method
""")

payment_diff = spark.sql(f"""
    (SELECT payment_method, transaction_count, successful_count, success_rate, total_amount_usd FROM recon_payment
     EXCEPT
     SELECT payment_method, transaction_count, successful_count, success_rate, total_amount_usd FROM {fq('gold_payment_success')})
    UNION ALL
    (SELECT payment_method, transaction_count, successful_count, success_rate, total_amount_usd FROM {fq('gold_payment_success')}
     EXCEPT
     SELECT payment_method, transaction_count, successful_count, success_rate, total_amount_usd FROM recon_payment)
""").count()

payment_ok = (payment_diff == 0)
all_passed = all_passed and payment_ok
print(f"\n--- gold_payment_success ---")
print(f"  Rebuilt rows:  {spark.sql('SELECT COUNT(*) FROM recon_payment').collect()[0][0]:,}")
print(f"  Gold rows:     {spark.table(fq('gold_payment_success')).count():,}")
print(f"  Row-by-row match: {'\u2713 PASS' if payment_ok else '\u2717 FAIL (' + str(payment_diff) + ')'}")
if not payment_ok:
    raise AssertionError("gold_payment_success reconciliation FAILED")

# --- 3d. gold_top_customers reconciliation ---
spark.sql("""
CREATE OR REPLACE TEMP VIEW recon_top_customers AS
SELECT * FROM (
    SELECT customer_id,
        CAST(SUM(eligible_revenue) AS DECIMAL(28,2)) AS eligible_spend,
        CAST(SUM(net_revenue) AS DECIMAL(28,2)) AS net_spend,
        CAST(SUM(CASE WHEN is_eligible THEN 1 ELSE 0 END) AS BIGINT) AS eligible_order_count,
        RANK() OVER (ORDER BY SUM(eligible_revenue) DESC) AS spend_rank
    FROM recon_order_revenue WHERE customer_id IS NOT NULL GROUP BY customer_id
    HAVING SUM(eligible_revenue) > 0
) WHERE spend_rank <= 100
""")

topc_diff = spark.sql(f"""
    (SELECT customer_id, eligible_spend, net_spend, eligible_order_count, spend_rank FROM recon_top_customers
     EXCEPT
     SELECT customer_id, eligible_spend, net_spend, eligible_order_count, spend_rank FROM {fq('gold_top_customers')})
    UNION ALL
    (SELECT customer_id, eligible_spend, net_spend, eligible_order_count, spend_rank FROM {fq('gold_top_customers')}
     EXCEPT
     SELECT customer_id, eligible_spend, net_spend, eligible_order_count, spend_rank FROM recon_top_customers)
""").count()

topc_ok = (topc_diff == 0)
all_passed = all_passed and topc_ok
print(f"\n--- gold_top_customers ---")
print(f"  Rebuilt rows:  {spark.sql('SELECT COUNT(*) FROM recon_top_customers').collect()[0][0]:,}")
print(f"  Gold rows:     {spark.table(fq('gold_top_customers')).count():,}")
print(f"  Row-by-row match: {'\u2713 PASS' if topc_ok else '\u2717 FAIL (' + str(topc_diff) + ')'}")
if not topc_ok:
    raise AssertionError("gold_top_customers reconciliation FAILED")

# --- 3e. Cross-mart consistency checks ---
print(f"\n--- Cross-mart consistency ---")

silver_txn_cnt = spark.table(fq("silver_transactions")).count()

daily_total_orders = spark.sql(f"SELECT CAST(SUM(order_count) AS BIGINT) FROM {fq('gold_daily_sales')}").collect()[0][0]
cross1_ok = (daily_total_orders == silver_orders_cnt)
all_passed = all_passed and cross1_ok
print(f"  daily_sales SUM(order_count) = {daily_total_orders:,} == silver_orders ({silver_orders_cnt:,}):  {'\u2713 PASS' if cross1_ok else '\u2717 FAIL'}")

daily_total_rev = spark.sql(f"SELECT SUM(eligible_revenue) FROM {fq('gold_daily_sales')}").collect()[0][0]
breakdown_total_rev = spark.sql(f"SELECT SUM(eligible_revenue) FROM {fq('gold_revenue_breakdown')}").collect()[0][0]
cross2_ok = (daily_total_rev == breakdown_total_rev)
all_passed = all_passed and cross2_ok
print(f"  daily_sales SUM(eligible_revenue) = {daily_total_rev:,.2f} == breakdown = {breakdown_total_rev:,.2f}:  {'\u2713 PASS' if cross2_ok else '\u2717 FAIL'}")

daily_total_net = spark.sql(f"SELECT SUM(net_revenue) FROM {fq('gold_daily_sales')}").collect()[0][0]
breakdown_total_net = spark.sql(f"SELECT SUM(net_revenue) FROM {fq('gold_revenue_breakdown')}").collect()[0][0]
cross3_ok = (daily_total_net == breakdown_total_net)
all_passed = all_passed and cross3_ok
print(f"  daily_sales SUM(net_revenue) = {daily_total_net:,.2f} == breakdown = {breakdown_total_net:,.2f}:  {'\u2713 PASS' if cross3_ok else '\u2717 FAIL'}")

payment_total_txn = spark.sql(f"SELECT CAST(SUM(transaction_count) AS BIGINT) FROM {fq('gold_payment_success')}").collect()[0][0]
cross4_ok = (payment_total_txn == silver_txn_cnt)
all_passed = all_passed and cross4_ok
print(f"  payment_success SUM(transaction_count) = {payment_total_txn:,} == silver_transactions ({silver_txn_cnt:,}):  {'\u2713 PASS' if cross4_ok else '\u2717 FAIL'}")

breakdown_total_orders = spark.sql(f"SELECT CAST(SUM(order_count) AS BIGINT) FROM {fq('gold_revenue_breakdown')}").collect()[0][0]
cross5_ok = (breakdown_total_orders == silver_orders_cnt)
all_passed = all_passed and cross5_ok
print(f"  breakdown SUM(order_count) = {breakdown_total_orders:,} == silver_orders ({silver_orders_cnt:,}):  {'\u2713 PASS' if cross5_ok else '\u2717 FAIL'}")

if not all_passed:
    raise AssertionError("Cross-mart consistency checks FAILED")

# --- Verified baseline figures ---
eligible_order_total = spark.sql(f"SELECT CAST(SUM(eligible_order_count) AS BIGINT) FROM {fq('gold_daily_sales')}").collect()[0][0]
cancelled_order_total = spark.sql(f"SELECT CAST(SUM(cancelled_order_count) AS BIGINT) FROM {fq('gold_daily_sales')}").collect()[0][0]
successful_txn_total = spark.sql(f"SELECT CAST(SUM(successful_count) AS BIGINT) FROM {fq('gold_payment_success')}").collect()[0][0]

print(f"\n{'=' * 70}")
print("VERIFIED BASELINE FIGURES (Databricks SQL)")
print(f"{'=' * 70}")
print(f"  Total orders (all statuses):       {daily_total_orders:,}")
print(f"  Total eligible orders (shipped):    {eligible_order_total:,}")
print(f"  Total cancelled orders:             {cancelled_order_total:,}")
print(f"  Total transactions:                 {silver_txn_cnt:,}")
print(f"  Total successful transactions:      {successful_txn_total:,}")
print(f"  Gross eligible revenue (USD):      ${daily_total_rev:,.2f}")
print(f"  Net revenue (USD):                 ${daily_total_net:,.2f}")
print(f"  Gold daily_sales rows:              {spark.table(fq('gold_daily_sales')).count():,}")
print(f"  Gold revenue_breakdown rows:        {spark.table(fq('gold_revenue_breakdown')).count():,}")
print(f"  Gold payment_success rows:          {spark.table(fq('gold_payment_success')).count():,}")
print(f"  Gold top_customers rows:            {spark.table(fq('gold_top_customers')).count():,}")

In [0]:
print("=" * 70)
print("END-TO-END RECONCILIATION SUMMARY")
print("=" * 70)
print(f"""
  Phase 1 — Source \u2192 Bronze parity:     \u2713 PASS  (2/2 datasets: 1M rows each)
  Phase 2 — Silver accounting invariant:   \u2713 PASS  (2/2 datasets: zero overlap, full coverage)
  Phase 3 — Gold \u2192 Silver reconciliation:   \u2713 PASS  (4/4 marts row-by-row match)
    \u00b7 gold_daily_sales:       {spark.table(fq('gold_daily_sales')).count():,} rows matched
    \u00b7 gold_revenue_breakdown: {spark.table(fq('gold_revenue_breakdown')).count():,} rows matched
    \u00b7 gold_payment_success:   {spark.table(fq('gold_payment_success')).count():,} rows matched
    \u00b7 gold_top_customers:     {spark.table(fq('gold_top_customers')).count():,} rows matched
    \u00b7 Cross-mart consistency:  5/5 checks passed

  Verified baseline (Databricks SQL):
    Gross eligible revenue:  ${daily_total_rev:,.2f}
    Net revenue:             ${daily_total_net:,.2f}
    Total orders:            {daily_total_orders:,}
    Total transactions:      {silver_txn_cnt:,}
""")
print("=" * 70)
print("\u2713 End-to-end reconciliation complete — all records and dollar amounts accounted for.")
print("=" * 70)